# Compensation History Validation

This notebook validates compensation records for the complete 10,000-employee synthetic workforce.

The main rules are:

- Every employee has exactly one hire compensation record.
- The hire compensation date matches the employee's hire date.
- Compensation dates do not occur before hiring or after employment ends.
- Compensation IDs are complete and unique.
- Base salaries do not decrease.
- Salaries remain inside the allowed job-role range.
- Bonus targets and equity values remain inside valid ranges.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

job_roles = pd.read_csv(
    RAW_DATA_DIR / "job_roles.csv"
)

compensation_history = pd.read_csv(
    RAW_DATA_DIR / "compensation_history.csv",
    parse_dates=["effective_date"],
)

print("Employees:", employees.shape)
print(
    "Compensation history:",
    compensation_history.shape,
)

Employees: (10000, 15)
Compensation history: (30997, 7)


## 1. Initial inspection

In [2]:
compensation_history.head(10)

,compensation_id,employee_id,effective_date,base_salary,bonus_target,equity_value,change_reason
0,500001,100001,2022-10-17,187000,26.0,92000,Hire
1,500002,100001,2023-10-17,192600,26.0,97500,Annual Review
2,500003,100001,2024-10-17,198000,26.0,99500,Annual Review
3,500004,100001,2025-10-17,204600,26.5,101500,Annual Review
4,500005,100002,2022-04-03,179400,23.0,56000,Hire
5,500006,100002,2023-04-03,186700,23.0,60000,Annual Review
6,500007,100002,2024-04-03,196600,23.5,62500,Annual Review
7,500008,100002,2025-04-03,203000,23.5,64000,Annual Review
8,500009,100002,2026-04-03,205200,23.5,67500,Annual Review
9,500010,100003,2021-06-28,181500,23.0,51000,Hire


In [3]:
compensation_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 30997 entries, 0 to 30996
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   compensation_id  30997 non-null  int64         
 1   employee_id      30997 non-null  int64         
 2   effective_date   30997 non-null  datetime64[us]
 3   base_salary      30997 non-null  int64         
 4   bonus_target     30997 non-null  float64       
 5   equity_value     30997 non-null  int64         
 6   change_reason    30997 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(1)
memory usage: 1.7 MB


## 2. Employee coverage

In [4]:
records_per_employee = (
    compensation_history
    .groupby("employee_id")
    .size()
    .rename("record_count")
)

records_per_employee.describe()

count    10000.000000
mean         3.099700
std          1.605808
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          6.000000
Name: record_count, dtype: float64

In [5]:
print(
    "Employees represented:",
    records_per_employee.size,
)

print(
    "Minimum records for one employee:",
    records_per_employee.min(),
)

print(
    "Maximum records for one employee:",
    records_per_employee.max(),
)

Employees represented: 10000
Minimum records for one employee: 1
Maximum records for one employee: 6


## 3. Hire-record checks

In [6]:
sorted_history = (
    compensation_history
    .sort_values(
        [
            "employee_id",
            "effective_date",
        ]
    )
)

first_records = (
    sorted_history
    .groupby(
        "employee_id",
        as_index=False,
    )
    .first()
    .merge(
        employees[
            [
                "employee_id",
                "hire_date",
            ]
        ],
        on="employee_id",
        how="left",
    )
)

hire_record_checks = pd.Series(
    {
        "every employee is represented": (
            first_records["employee_id"]
            .nunique()
            == len(employees)
        ),
        "first reason is Hire": (
            first_records[
                "change_reason"
            ].eq("Hire").all()
        ),
        "first date matches hire date": (
            first_records[
                "effective_date"
            ].eq(
                first_records[
                    "hire_date"
                ]
            ).all()
        ),
    },
    name="passed",
)

hire_record_checks

every employee is represented    True
first reason is Hire             True
first date matches hire date     True
Name: passed, dtype: bool

## 4. Compensation-date checks

In [7]:
employee_dates = employees[
    [
        "employee_id",
        "hire_date",
        "termination_date",
    ]
].copy()

employee_dates[
    "employment_end_date"
] = employee_dates[
    "termination_date"
].fillna(
    pd.Timestamp("2026-06-30")
)

compensation_dates = (
    compensation_history
    .merge(
        employee_dates,
        on="employee_id",
        how="left",
    )
)

date_checks = pd.Series(
    {
        "no record is before hire": (
            compensation_dates[
                "effective_date"
            ]
            .ge(
                compensation_dates[
                    "hire_date"
                ]
            )
            .all()
        ),
        "no record is after employment": (
            compensation_dates[
                "effective_date"
            ]
            .le(
                compensation_dates[
                    "employment_end_date"
                ]
            )
            .all()
        ),
        "employee-date combinations are unique": (
            not compensation_history
            .duplicated(
                subset=[
                    "employee_id",
                    "effective_date",
                ]
            )
            .any()
        ),
    },
    name="passed",
)

date_checks

no record is before hire                 True
no record is after employment            True
employee-date combinations are unique    True
Name: passed, dtype: bool

## 5. Salary-band checks

In [8]:
compensation_details = (
    compensation_history
    .merge(
        employees[
            [
                "employee_id",
                "job_role_id",
                "department_id",
                "organizational_level",
            ]
        ],
        on="employee_id",
        how="left",
    )
    .merge(
        job_roles[
            [
                "job_role_id",
                "job_title",
                "salary_band_min",
                "salary_band_max",
            ]
        ],
        on="job_role_id",
        how="left",
    )
)

compensation_details[
    "allowed_salary_max"
] = (
    compensation_details[
        "salary_band_max"
    ]
    * 1.08
)

salary_band_checks = pd.Series(
    {
        "all salaries are positive": (
            compensation_details[
                "base_salary"
            ].gt(0).all()
        ),
        "no salary is below the band": (
            compensation_details[
                "base_salary"
            ]
            .ge(
                compensation_details[
                    "salary_band_min"
                ]
            )
            .all()
        ),
        "no salary exceeds the allowed cap": (
            compensation_details[
                "base_salary"
            ]
            .le(
                compensation_details[
                    "allowed_salary_max"
                ]
            )
            .all()
        ),
        "bonus targets are between 0 and 40": (
            compensation_details[
                "bonus_target"
            ].between(
                0,
                40,
            ).all()
        ),
        "equity values are nonnegative": (
            compensation_details[
                "equity_value"
            ].ge(0).all()
        ),
    },
    name="passed",
)

salary_band_checks

all salaries are positive             True
no salary is below the band           True
no salary exceeds the allowed cap     True
bonus targets are between 0 and 40    True
equity values are nonnegative         True
Name: passed, dtype: bool

## 6. Salary progression

In [9]:
salary_progression = (
    compensation_history
    .sort_values(
        [
            "employee_id",
            "effective_date",
        ]
    )
    .copy()
)

salary_progression[
    "previous_salary"
] = (
    salary_progression
    .groupby("employee_id")[
        "base_salary"
    ]
    .shift(1)
)

salary_progression[
    "salary_change"
] = (
    salary_progression[
        "base_salary"
    ]
    - salary_progression[
        "previous_salary"
    ]
)

salary_decreases = salary_progression[
    salary_progression[
        "salary_change"
    ] < 0
]

print(
    "Number of salary decreases:",
    len(salary_decreases),
)

Number of salary decreases: 0


## 7. Compensation change reasons

In [10]:
change_reason_summary = (
    compensation_history[
        "change_reason"
    ]
    .value_counts()
    .rename_axis("change_reason")
    .reset_index(name="record_count")
)

change_reason_summary[
    "record_percent"
] = (
    change_reason_summary[
        "record_count"
    ]
    / len(compensation_history)
    * 100
).round(2)

change_reason_summary

,change_reason,record_count,record_percent
0,Annual Review,16742,54.01
1,Hire,10000,32.26
2,Promotion,2146,6.92
3,Market Adjustment,2109,6.80


In [11]:
hire_record_count = (
    compensation_history[
        "change_reason"
    ]
    .eq("Hire")
    .sum()
)

print(
    "Hire records:",
    hire_record_count,
)

Hire records: 10000


## 8. Latest employee compensation

In [12]:
latest_compensation = (
    compensation_history
    .sort_values(
        [
            "employee_id",
            "effective_date",
        ]
    )
    .groupby("employee_id")
    .tail(1)
)

latest_compensation_details = (
    latest_compensation
    .merge(
        employees[
            [
                "employee_id",
                "department_id",
                "organizational_level",
            ]
        ],
        on="employee_id",
        how="left",
    )
)

latest_compensation_summary = (
    latest_compensation_details
    .groupby(
        "organizational_level"
    )
    .agg(
        employee_count=(
            "employee_id",
            "count",
        ),
        average_base_salary=(
            "base_salary",
            "mean",
        ),
        median_base_salary=(
            "base_salary",
            "median",
        ),
        average_bonus_target=(
            "bonus_target",
            "mean",
        ),
        average_equity_value=(
            "equity_value",
            "mean",
        ),
    )
    .round(2)
)

latest_compensation_summary

,employee_count,average_base_salary,median_base_salary,average_bonus_target,average_equity_value
organizational_level,,,,,
Department Head,8,127050.00,124200.0,23.25,43375.00
Individual Contributor,9158,90267.58,86100.0,8.71,5671.11
Senior Manager,68,132604.41,121850.0,19.41,33191.18
Team Manager,766,129358.22,118200.0,15.67,20663.84


## 9. Complete validation summary

In [13]:
basic_checks = pd.Series(
    {
        "table has seven columns": (
            len(
                compensation_history.columns
            )
            == 7
        ),
        "compensation IDs are complete": (
            compensation_history[
                "compensation_id"
            ].notna().all()
        ),
        "compensation IDs are unique": (
            compensation_history[
                "compensation_id"
            ].is_unique
        ),
        "all 10,000 employees are represented": (
            compensation_history[
                "employee_id"
            ].nunique()
            == 10_000
        ),
        "exactly 10,000 hire records exist": (
            hire_record_count
            == 10_000
        ),
        "no salaries decrease": (
            len(salary_decreases)
            == 0
        ),
    },
    name="passed",
)

all_checks = pd.concat(
    [
        basic_checks,
        hire_record_checks,
        date_checks,
        salary_band_checks,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,table has seven columns,True
1,compensation IDs are complete,True
2,compensation IDs are unique,True
3,"all 10,000 employees are represented",True
4,"exactly 10,000 hire records exist",True
5,no salaries decrease,True
6,every employee is represented,True
7,first reason is Hire,True
8,first date matches hire date,True
9,no record is before hire,True


In [14]:
if validation_results["passed"].all():
    print(
        "All compensation-history "
        "validation checks passed."
    )
else:
    print(
        "One or more compensation-history "
        "validation checks failed."
    )

All compensation-history validation checks passed.


## 10. Conclusions

The synthetic compensation-history table successfully represents compensation changes for all 10,000 employees.

### Successful checks

- Every employee has exactly one hire compensation record.
- Hire compensation dates match employee hire dates.
- Compensation records remain inside each employee's employment period.
- Compensation IDs are complete and unique.
- Base salaries do not decrease.
- Salaries remain inside the allowed role-band range.
- Bonus targets remain between 0% and 40%.
- Equity values are nonnegative.
- All compensation-history validation checks passed.

### Current simplifications

- Base salary is annualized for both salaried and hourly employees.
- Historical compensation uses the employee's current job-role salary band.
- Promotion compensation records do not yet create matching employee-event records.
- Performance ratings are not yet used to determine annual increases.
- Bonus and equity values are synthetic estimates rather than real company policies.